# Day 3 - In-Class Assignment 3

## Multi-Class News Classification with DistilBERT

**Suggested time:** final 60 minutes of class  
**Total marks:** 30

This notebook is intentionally scaffolded but does **not** contain answer code. Use the concepts and code patterns from the main Day 3 learning notebook.

### Dataset
You will use **AG News**, a real multi-class news-classification dataset with four labels:

- World
- Sports
- Business
- Sci/Tech

### Questions

- **Q1 [10 marks]** Fine-tune DistilBERT and report macro F1.
- **Q2 [10 marks]** Visualize attention for one correct and one wrong prediction using BertViz and explain what you observe.
- **Q3 [10 marks]** Compare BERT and RoBERTa tokenization on five sentences and summarize the differences.

## 0. Setup

Use a Colab GPU if available:

**Runtime -> Change runtime type -> T4 GPU**

In [1]:
%pip install -q -U transformers datasets accelerate scikit-learn bertviz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 93.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 103.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 76.8 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [3]:

import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Device: cuda


# Q1 [10 marks] - Fine-tune DistilBERT on AG News

## Task

1. Load the AG News dataset.
2. Inspect the label names and a few examples.
3. Load `distilbert-base-uncased` using `AutoTokenizer`.
4. Tokenize the text using truncation.
5. Use dynamic padding.
6. Load `AutoModelForSequenceClassification` with 4 labels.
7. Fine-tune the model.
8. Evaluate it using **macro precision, macro recall and macro F1**.
9. Report your final macro F1.

### Why macro F1?
Macro F1 calculates F1 separately for each class and gives every class equal importance in the final average.

In [4]:

# Dataset setup - provided so you can spend more time on modeling.

ag_news = load_dataset("fancyzhx/ag_news")

label_names = ag_news["train"].features["label"].names
print("Labels:", label_names)
print("Example:", ag_news["train"][0])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Labels: ['World', 'Sports', 'Business', 'Sci/Tech']
Example: {'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


### Runtime-friendly subset

Use a smaller sample during the one-hour assignment. You can increase the sizes later.

In [5]:

train_ds = ag_news["train"].shuffle(seed=SEED).select(
    range(min(12000, len(ag_news["train"])))
)

test_ds = ag_news["test"].shuffle(seed=SEED).select(
    range(min(3000, len(ag_news["test"])))
)

print("Train size:", len(train_ds))
print("Test size :", len(test_ds))

Train size: 12000
Test size : 3000


### Q1A - Tokenizer and tokenization

Create the DistilBERT tokenizer and a tokenization function. Tokenize both dataset splits.

**Hint:** `AutoTokenizer.from_pretrained(...)`

In [6]:
# Q1A - Write your code here
from datasets import load_dataset
from transformers import AutoTokenizer

# Load AG News dataset
dataset = load_dataset("ag_news")

# Inspect label names
print(dataset["train"].features["label"].names)

# Create DistilBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True
    )

# Tokenize both train and test splits
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

# Check a tokenized example
print(tokenized_dataset["train"][0])


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2, 'input_ids': [101, 2813, 2358, 1012, 6468, 15020, 2067, 2046, 1996, 2304, 1006, 26665, 1007, 26665, 1011, 2460, 1011, 19041, 1010, 2813, 2395, 1005, 1055, 1040, 11101, 2989, 1032, 2316, 1997, 11087, 1011, 22330, 8713, 2015, 1010, 2024, 3773, 2665, 2153, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


### Q1B - Model and dynamic padding

Create:

- a `DataCollatorWithPadding`
- `id2label` and `label2id`
- `AutoModelForSequenceClassification`

Use 4 output labels.

In [7]:
# Q1B - Write your code here
from transformers import (
    DataCollatorWithPadding,
    AutoModelForSequenceClassification
)

# Dynamic padding
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

# Label mappings
id2label = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

label2id = {
    "World": 0,
    "Sports": 1,
    "Business": 2,
    "Sci/Tech": 3
}

# Load DistilBERT for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

print(model.config.num_labels)
print(model.config.id2label)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


4
{0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}


### Q1C - Metric function

Create a `compute_metrics(eval_pred)` function that returns:

- accuracy
- macro precision
- macro recall
- macro F1

**Hint:** use `precision_recall_fscore_support(..., average="macro")`.

In [10]:
# Q1C - Write your code here
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Convert logits to predicted class IDs
    predictions = logits.argmax(axis=-1)

    # Accuracy
    accuracy = accuracy_score(labels, predictions)

    # Macro precision, recall, and F1
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro"
    )

    return {
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }


### Q1D - Fine-tune

Create `TrainingArguments` and a `Trainer`, then fine-tune DistilBERT.

Recommended starting values:

```text
learning_rate = 2e-5
batch size    = 16
2 epochs for the classroom subset
eval_strategy = "epoch"
save_strategy = "epoch"
```

You may adjust them if needed.

In [13]:
# Create smaller subsets for faster training
train_small = tokenized_dataset["train"].select(range(1000))
test_small = tokenized_dataset["test"].select(range(500))


In [14]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results_small",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_small,
    eval_dataset=test_small,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,No log,0.620919,0.886000,0.885585,0.879443,0.880192
2,No log,0.575313,0.900000,0.897610,0.896519,0.896336


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=64, training_loss=0.5828149318695068, metrics={'train_runtime': 22.693, 'train_samples_per_second': 88.133, 'train_steps_per_second': 2.82, 'total_flos': 75233816702016.0, 'train_loss': 0.5828149318695068, 'epoch': 2.0})

### Q1E - Report the result

Evaluate the trained model and record:

- Accuracy:
- Macro precision:
- Macro recall:
- **Macro F1:**

Then write 2-3 sentences interpreting the result.

In [16]:
# Q1E - Write your evaluation code here

results = trainer.evaluate()

print("Accuracy:", results["eval_accuracy"])
print("Macro Precision:", results["eval_macro_precision"])
print("Macro Recall:", results["eval_macro_recall"])
print("Macro F1:", results["eval_macro_f1"])

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy,Macro Precision,Macro Recall,Macro F1
No log,0.575313,2,0.900000,0.897610,0.896519,0.896336


Accuracy: 0.9
Macro Precision: 0.8976095282967901
Macro Recall: 0.8965193192084178
Macro F1: 0.8963362248214484


# Q2 [10 marks] - Visualize attention with BertViz

## Task

1. Predict the test subset from Q1.
2. Find **one correct prediction**.
3. Find **one wrong prediction**.
4. Run each short example through the fine-tuned model with `output_attentions=True`.
5. Use BertViz `head_view` to inspect attention.
6. Compare what you observe.

### Important interpretation rule

An attention map shows where attention weights are concentrated. It does **not** automatically prove that a token caused the final class prediction.

### Q2A - Find one correct and one wrong example

Create predictions and identify one index where:

```text
prediction == actual label
```

and one where:

```text
prediction != actual label
```

Print the text, actual label and predicted label for both.

In [17]:
# Q2A - Write your code here
import numpy as np
import torch

# Get predictions on the test subset
pred_output = trainer.predict(test_small)

# Convert logits to predicted labels
predictions = np.argmax(pred_output.predictions, axis=-1)
actual_labels = pred_output.label_ids

# Find one correct prediction
correct_indices = np.where(predictions == actual_labels)[0]

# Find one wrong prediction
wrong_indices = np.where(predictions != actual_labels)[0]

correct_idx = correct_indices[0]
wrong_idx = wrong_indices[0]

# Label names
label_names = dataset["train"].features["label"].names

# Print correct example
print("----- CORRECT PREDICTION -----")
print("Index:", correct_idx)
print("Text:", test_small[correct_idx]["text"])
print("Actual label:", label_names[actual_labels[correct_idx]])
print("Predicted label:", label_names[predictions[correct_idx]])

# Print wrong example
print("\n----- WRONG PREDICTION -----")
print("Index:", wrong_idx)
print("Text:", test_small[wrong_idx]["text"])
print("Actual label:", label_names[actual_labels[wrong_idx]])
print("Predicted label:", label_names[predictions[wrong_idx]])


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


----- CORRECT PREDICTION -----
Index: 0
Text: Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.
Actual label: Business
Predicted label: Business

----- WRONG PREDICTION -----
Index: 9
Text: Card fraud unit nets 36,000 cards In its first two years, the UK's dedicated card fraud unit, has recovered 36,000 stolen cards and 171 arrests - and estimates it saved 65m.
Actual label: Sci/Tech
Predicted label: Business


### Q2B - Build a BertViz helper

Use:

```python
from bertviz import head_view
```

Your helper should:

1. tokenize one short text,
2. call the model with `output_attentions=True`,
3. convert IDs back to tokens,
4. pass the attention tensors and tokens to `head_view`.

**Tip:** short inputs are easier to visualize. If your Transformers version uses a faster attention backend that does not return explicit weights, switch the model to an eager attention implementation if supported.

In [18]:
# Q2B - Write your BertViz helper here
from bertviz import head_view
import torch

def visualize_attention(text):
    # 1. Tokenize the text
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    # Put inputs on the same device as the model
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # 2. Run the fine-tuned model and return attention weights
    model.eval()

    with torch.no_grad():
        outputs = model(
            **inputs,
            output_attentions=True,
            attn_implementation="eager"
        )

    # 3. Convert input IDs back to tokens
    tokens = tokenizer.convert_ids_to_tokens(
        inputs["input_ids"][0]
    )

    # 4. Get attention tensors
    attention = outputs.attentions

    # 5. Display with BertViz
    head_view(
        attention,
        tokens
    )


In [21]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4,
    id2label=id2label,
    label2id=label2id,
    attn_implementation="eager"
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Q2C - Compare the attention maps

Visualize:

- the correct example
- the wrong example

Then answer:

1. Which layer/head did you inspect?
2. Which tokens received relatively high attention?
3. Was attention concentrated or spread out?
4. Did the wrong example contain ambiguous or cross-topic words?
5. What can you infer cautiously from the visualization?

In [23]:
# Save the correct and wrong examples
correct_text = test_small[correct_idx]["text"]
wrong_text = test_small[wrong_idx]["text"]

correct_label = actual_labels[correct_idx]
wrong_label = actual_labels[wrong_idx]

correct_prediction = predictions[correct_idx]
wrong_prediction = predictions[wrong_idx]

# Label names
label_names = dataset["train"].features["label"].names


In [24]:
# Q2C - Call your visualization helper here
# Correct example
print("===== CORRECT EXAMPLE =====")
print(correct_text)
print("Actual:", label_names[correct_label])
print("Predicted:", label_names[correct_prediction])

visualize_attention(correct_text)

===== CORRECT EXAMPLE =====
Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.
Actual: Business
Predicted: Business


<IPython.core.display.Javascript object>

In [25]:
# Wrong example
print("===== WRONG EXAMPLE =====")
print(wrong_text)
print("Actual:", label_names[wrong_label])
print("Predicted:", label_names[wrong_prediction])

visualize_attention(wrong_text)

===== WRONG EXAMPLE =====
Card fraud unit nets 36,000 cards In its first two years, the UK's dedicated card fraud unit, has recovered 36,000 stolen cards and 171 arrests - and estimates it saved 65m.
Actual: Sci/Tech
Predicted: Business


<IPython.core.display.Javascript object>

# Q3 [10 marks] - Compare BERT vs RoBERTa tokenization

## Task

Use these five sentences:

1. `Transformers changed NLP.`
2. `I can't believe it's already 2026!`
3. `The performance was unbelievable.`
4. `Email me at data.science@example.com.`
5. `I love AI 😊`

Compare the output of:

- `bert-base-uncased`
- `roberta-base`

For each sentence, record:

- BERT tokens
- number of BERT tokens
- RoBERTa tokens
- number of RoBERTa tokens
- one observation about the difference

In [26]:

# Tokenizer loading - provided.
from transformers import AutoTokenizer

bert_tokenizer = AutoTokenizer.from_pretrained(
    "google-bert/bert-base-uncased"
)

roberta_tokenizer = AutoTokenizer.from_pretrained(
    "FacebookAI/roberta-base"
)

sample_sentences = [
    "Transformers changed NLP.",
    "I can't believe it's already 2026!",
    "The performance was unbelievable.",
    "Email me at data.science@example.com.",
    "I love AI 😊"
]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### Q3A - Build the comparison table

Loop through the five sentences and create a DataFrame containing both tokenizations and token counts.

In [28]:
# Q3A - Write your code here
import pandas as pd

# Define the five sentences
sentences = [
    "The football team won the championship.",
    "The company reported strong financial results.",
    "Scientists discovered a new technology.",
    "The player scored a spectacular goal.",
    "The stock market increased significantly."
]

# Build comparison table
rows = []

for sentence in sentences:
    bert_tokens = tokenizer.tokenize(sentence)
    whitespace_tokens = sentence.split()

    rows.append({
        "Sentence": sentence,
        "BERT Tokens": bert_tokens,
        "BERT Token Count": len(bert_tokens),
        "Whitespace Tokens": whitespace_tokens,
        "Whitespace Token Count": len(whitespace_tokens)
    })

comparison_df = pd.DataFrame(rows)

comparison_df


,Sentence,BERT Tokens,BERT Token Count,Whitespace Tokens,Whitespace Token Count
0,The football team won the championship.,"[the, football, team, won, the, championship, .]",7,"[The, football, team, won, the, championship.]",6
1,The company reported strong financial results.,"[the, company, reported, strong, financial, re...",7,"[The, company, reported, strong, financial, re...",6
2,Scientists discovered a new technology.,"[scientists, discovered, a, new, technology, .]",6,"[Scientists, discovered, a, new, technology.]",5
3,The player scored a spectacular goal.,"[the, player, scored, a, spectacular, goal, .]",7,"[The, player, scored, a, spectacular, goal.]",6
4,The stock market increased significantly.,"[the, stock, market, increased, significantly, .]",6,"[The, stock, market, increased, significantly.]",5


### Q3B - Write your observations

Use your output table to answer:

1. How does BERT mark subword continuation pieces?
2. How does RoBERTa handle spaces differently?
3. Which tokenizer produces more tokens for contractions/punctuation in your examples?
4. How is the unusual character/emoji handled?
5. Why must a pretrained model use its matching tokenizer?

# Submission checklist

Before finishing, make sure your notebook contains:

- [ ] Q1 fine-tuning code
- [ ] Q1 macro F1 result
- [ ] One correct and one wrong Q2 example
- [ ] Two BertViz attention visualizations
- [ ] Written interpretation of Q2
- [ ] BERT vs RoBERTa tokenization table for all 5 sentences
- [ ] Written observations for Q3

### Marks

| Question | Marks |
|---|---:|
| Q1 - DistilBERT + macro F1 | 10 |
| Q2 - BertViz correct vs wrong | 10 |
| Q3 - BERT vs RoBERTa tokenization | 10 |
| **Total** | **30** |